# Pose Suggestion Camera Assistant — notebook điều khiển (v2)

**Nguyên tắc:** code nằm trong các file `.py` ở `pose-assistant/code/` trên Drive; notebook này chỉ **import và gọi hàm**.
Không dán code vào ô và không dùng "Fix with Gemini" (từng ghi đè file thật bằng bản mock).

Các file cần có trong `pose-assistant/code/`: `02_three_models.py`, `05_upgrade_v2.py`, `06_export_coreml.py`, `07_build_templates_mpii.py`.
Chạy lần lượt từ trên xuống.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cài thư viện (chạy 1 lần mỗi phiên) + tải model MediaPipe
!pip install mediapipe pycocotools torchvision --break-system-packages -q
!wget -O /content/pose_landmarker.task -q https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task
import os; os.chdir('/content')

In [ ]:
import sys, os, json, importlib
BASE = '/content/drive/MyDrive/pose-assistant'
CODE = f'{BASE}/code'
sys.path.insert(0, CODE)

need = ['02_three_models.py', '05_upgrade_v2.py', '06_export_coreml.py', '07_build_templates_mpii.py']
missing = [f for f in need if not os.path.exists(f'{CODE}/{f}')]
print('File code còn thiếu trong', CODE, ':', missing or 'không thiếu')

mat  = f'{BASE}/mpii_dataset/mpii_human_pose_v1_u12_2/mpii_human_pose_v1_u12_1.mat'
imgs = f'{BASE}/mpii_dataset/images'
ckpt = f'{BASE}/checkpoints/scene_analyzer.pt'
tpl  = f'{BASE}/templates/pose_templates_v1.json'
print('MPII .mat   :', os.path.exists(mat))
print('MPII ảnh    :', len(os.listdir(imgs)) if os.path.isdir(imgs) else 0, 'file')
print('Checkpoint  :', os.path.exists(ckpt))
if os.path.exists(tpl):
    t = json.load(open(tpl, encoding='utf-8'))['templates']
    mock = any(str(x['template_id']).startswith('demo_') for x in t)
    print('Template    :', len(t), 'mục', '-> ĐANG LÀ BẢN MOCK, cần dựng template thật (bước 7-10)' if mock else '')
else:
    print('Template    : chưa có')

In [ ]:
# (Chỉ chạy nếu 'Checkpoint: False' ở ô trên) Train head SceneAnalyzer + benchmark + LƯU checkpoint. Cần GPU.
if not os.path.exists(ckpt):
    get_ipython().system(f'python {CODE}/02_three_models.py')
else:
    print('Đã có checkpoint, bỏ qua.')

In [ ]:
# Tự kiểm tra logic (không cần dữ liệu thật). Phải thấy 'TẤT CẢ TEST ... PASS'
m05 = importlib.reload(importlib.import_module('05_upgrade_v2'))
m07 = importlib.reload(importlib.import_module('07_build_templates_mpii'))
print('--- 05 ---'); m05.selftest()
print('--- 07 ---'); m07.selftest()

In [ ]:
# BƯỚC 7 — quét MPII, lọc ứng viên (lưu candidates.json lên Drive)
cands = m07.scan()

In [ ]:
# BƯỚC 8 — xem contact sheet, ghi lại ID ảnh bạn thích. Đổi range để xem thêm trang.
for p in range(0, 3):
    m07.sheet(p)

In [ ]:
# BƯỚC 9 — ĐIỀN LỰA CHỌN CỦA BẠN (15–20 mục). Các ID dưới đây chỉ là VÍ DỤ, hãy thay bằng ID bạn chọn ở bước 8.
# location: 'any' | 'indoor' | 'outdoor'.  delta_hint để trống: hệ thống tự sinh gợi ý chỉnh từ sai lệch thật.
SELECTION = [
    {'candidate_id': 0, 'pose_name': 'standing_basic',   'caption': 'Đứng thẳng, thả lỏng vai',  'location': 'any'},
    {'candidate_id': 1, 'pose_name': 'hand_on_hip',      'caption': 'Thử chống một tay lên hông', 'location': 'any'},
    {'candidate_id': 2, 'pose_name': 'arm_raised',       'caption': 'Giơ một tay lên nhẹ nhàng',  'location': 'outdoor'},
    # ... thêm tới 15–20 mục
]

In [ ]:
# BƯỚC 10 — ghi pose_templates_v1.json (tự sao lưu bản cũ; từ chối ghi đè nếu số template ít đi)
payload = m07.build(SELECTION)

In [ ]:
# BƯỚC 11 — chạy thử toàn pipeline trên 1 ảnh MPII (pose -> scene -> gợi ý -> ghost overlay)
import json
c0 = json.load(open(f'{BASE}/templates/mpii_work/candidates.json', encoding='utf-8'))
test_img = os.path.join(c0['img_dir'], c0['candidates'][5]['image_name'])
m05.run_source(test_img, '/content/ket_qua.jpg', tpl, ckpt, True)
from IPython.display import Image, display
display(Image('/content/ket_qua.jpg'))

In [ ]:
# BƯỚC 12 — export SceneAnalyzer sang Core ML cho iOS (convert được trên Colab; kiểm tra độ lệch cần chạy trên Mac)
!pip install coremltools -q
get_ipython().system(f'python {CODE}/06_export_coreml.py --ckpt {ckpt} --out {BASE}/checkpoints/SceneAnalyzer.mlpackage')

**Số liệu đã có để ghi vào README/CV (nhớ nêu đủ ngữ cảnh):**
- Pose (MediaPipe Lite, COCO val, ảnh 1 người): phát hiện được 159/200 ảnh (79.5%); PCK@0.5·torso = 0.842 *chỉ trên ảnh đã phát hiện được*.
- Scene (MobileNetV3-Small, chỉ train head, Places365 val): accuracy test ≈ 0.90 so với baseline 0.56.
- Latency: **chưa có** — phải đo trên iPhone thật.